# Observe repeated internal computation

Map calls to a reused module onto explicit tick and layer coordinates.

In [ ]:
import torch
from tensordict import TensorDict
from tensordict.nn import TensorDictModule
from xdrl import (
    BatchSemantics,
    InteractionContract,
    InteractionPhase,
    InternalComputationAxis,
    InternalComputationSemantics,
    InternalOccurrence,
    InternalOccurrenceSelection,
    KeyPresence,
    KeyRole,
    KeySchema,
    ModelRole,
    ObservationTrace,
    RecurrentSemantics,
    RecurrentStateTransition,
    RetentionPolicy,
    RuntimeInteractionContext,
    TensorDictSchema,
    TensorRetention,
)


class RepeatedCell(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.cell = torch.nn.Conv2d(1, 1, kernel_size=1, bias=False)
        torch.nn.init.constant_(self.cell.weight, 0.5)

    def forward(self, observation, state):
        hidden = state + observation
        for _tick in range(2):
            hidden = self.cell(hidden)
            hidden = self.cell(hidden)
        return hidden, hidden.mean(dim=(-2, -1))

In [ ]:
semantics = InternalComputationSemantics(
    axes=(
        InternalComputationAxis("tick", (0, 1)),
        InternalComputationAxis("layer", (0, 1)),
    ),
    occurrences=tuple(
        InternalOccurrence("module.cell", index, coordinates)
        for index, coordinates in enumerate(((0, 0), (0, 1), (1, 0), (1, 1)))
    ),
    recurrent_state_keys=(("state",), ("next", "state")),
)

batch_dims = BatchSemantics(("env",))
contract = InteractionContract(
    identity="repeated-cell:tutorial",
    role=ModelRole.ACTOR,
    phase=InteractionPhase.EVALUATION,
    module_path="policy",
    input_schema=TensorDictSchema(
        (
            KeySchema("observation", KeyRole.OBSERVATION, KeyPresence.REQUIRED),
            KeySchema("state", KeyRole.STATE, KeyPresence.REQUIRED),
            KeySchema("is_init", KeyRole.TERMINATION, KeyPresence.REQUIRED),
        ),
        batch_dims,
    ),
    output_schema=TensorDictSchema(
        (
            KeySchema(("next", "state"), KeyRole.STATE, KeyPresence.PRODUCED),
            KeySchema("action", KeyRole.ACTION, KeyPresence.PRODUCED),
        ),
        batch_dims,
    ),
    recurrent=RecurrentSemantics(
        transitions=(RecurrentStateTransition(("state",), ("next", "state")),),
        reset_keys=(("is_init",),),
    ),
    internal_computation=semantics,
)

In [ ]:
module = TensorDictModule(
    RepeatedCell(),
    in_keys=["observation", "state"],
    out_keys=[("next", "state"), "action"],
)
batch = TensorDict(
    {
        "observation": torch.ones(2, 1, 2, 2),
        "state": torch.zeros(2, 1, 2, 2),
        "is_init": torch.zeros(2, 1, dtype=torch.bool),
    },
    batch_size=[2],
)
trace = ObservationTrace(RetentionPolicy(tensor=TensorRetention.DETACHED))
interaction = RuntimeInteractionContext(contract, module, batch, observations=trace)

with interaction, interaction.observe_internal_computation():
    result = interaction.invoke(batch.clone())

records = [record for record in trace.records if record.raw_call_index is not None]
observed = [(record.raw_call_index, record.internal_coordinates) for record in records]
assert observed == [
    (0, (("tick", 0), ("layer", 0))),
    (1, (("tick", 0), ("layer", 1))),
    (2, (("tick", 1), ("layer", 0))),
    (3, (("tick", 1), ("layer", 1))),
]
observed

In [ ]:
second_tick = semantics.select(InternalOccurrenceSelection((("tick", 1),)))
selected = [(item.module_path, item.call_index) for item in second_tick]
assert selected == [("module.cell", 2), ("module.cell", 3)]
selected